# Warm-Orthogonal vs. Newton Alignment in a Tiny Deep Linear Network

This notebook is the analysis counterpart to `run_experiment.py` in the same directory. It deliberately **imports and runs the script's implementation** rather than duplicating the experimental core, so the notebook and script stay aligned.

## What is actually computed here?

At a set of SGD training checkpoints in a toy 2-layer 4×4 deep linear network, we compute:

- the full finite-difference Hessian,
- a **pseudoinverse Newton direction** `d_newton = -pinv(H) g`,
- the cosine similarity between `d_newton` and layerwise Newton–Schulz-transformed gradient directions for several iteration counts `k`,
- SGD and Adam **snapshot-direction** baselines on the **same SGD-trained trajectory**.

## Scope and non-claims

This notebook does **not** claim that the current metric:

- computes the natural gradient,
- proves gauge removal,
- proves gauge-null projection,
- or proves that `k=20` is exact orthogonalization.

Instead, it reports a narrow directional-alignment proxy in a tiny, fully inspectable toy model.


## Reporting standards for this first completion pass

Relative to the earlier version, this notebook now adds:

1. explicit notebook-safe path and import handling,
2. a reproducibility/config/runtime section,
3. structured use of `run_experiment()` results from the script,
4. summary tables and figures,
5. a small orthogonality-residual diagnostic for the Newton–Schulz iteration count,
6. a calibrated conclusion that states whether the current run supports or rejects the originally expected inverted-U story.

The script remains the **source of truth** for the computation; the notebook is the analysis and presentation layer.


In [ ]:
from pathlib import Path
import importlib.util
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown as _IPyMarkdown, display as _ipy_display
except ImportError:
    _IPyMarkdown = None
    _ipy_display = None


def display(obj):
    if _ipy_display is not None:
        _ipy_display(obj)
    elif hasattr(obj, 'to_string'):
        print(obj.to_string())
    else:
        print(obj)


def Markdown(text):
    return _IPyMarkdown(text) if _IPyMarkdown is not None else text


def show_figure(fig):
    """Display interactively when possible; otherwise close cleanly for batch smoke tests."""
    fig.tight_layout()
    backend = plt.get_backend().lower()
    if 'agg' in backend:
        plt.close(fig)
    else:
        plt.show()


plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)


In [ ]:
def find_repo_root(start=None):
    """Search upward from the working directory for the repository root."""
    start = Path.cwd() if start is None else Path(start)
    relative_target = Path('experiments/Tier_1_Core_Mechanism_Tests/WARM_ORTHO_newton_alignment/run_experiment.py')
    for candidate in [start, *start.parents]:
        if (candidate / relative_target).exists():
            return candidate
    raise FileNotFoundError(
        'Could not locate repository root by searching upward for '
        f"{relative_target} from {start}."
    )


REPO_ROOT = find_repo_root()
EXPERIMENT_DIR = REPO_ROOT / 'experiments' / 'Tier_1_Core_Mechanism_Tests' / 'WARM_ORTHO_newton_alignment'
MODULE_PATH = EXPERIMENT_DIR / 'run_experiment.py'

spec = importlib.util.spec_from_file_location('warm_ortho_newton_alignment', MODULE_PATH)
warm_ortho = importlib.util.module_from_spec(spec)
spec.loader.exec_module(warm_ortho)

print(f'Repository root: {REPO_ROOT}')
print(f'Experiment directory: {EXPERIMENT_DIR}')
print(f'Imported module: {MODULE_PATH.name}')
print(f'Available entrypoints: run_experiment={hasattr(warm_ortho, "run_experiment")}, main={hasattr(warm_ortho, "main")}')


## Reproducibility, runtime, and raw result objects

The next cell executes the default experiment through the script API. The returned object contains configuration, seeds, nested per-seed/per-step measurements, flattened measurement records, aggregate summaries, and a compact verdict block.


In [ ]:
analysis_start = time.perf_counter()
results = warm_ortho.run_experiment(verbose=False)
analysis_wall_time_sec = time.perf_counter() - analysis_start

config = results['config']
summary = results['summary']
verdict = results['verdict']
direction_metadata = results['direction_metadata']
direction_labels_by_k = direction_metadata['direction_labels_by_k']
selected_k_values = direction_metadata['selected_k_values_for_reporting']
baseline_labels = direction_metadata['baseline_labels']
records_df = pd.DataFrame(results['measurement_records'])

if records_df.empty:
    raise RuntimeError('No measurement records were produced; the notebook cannot continue.')

print(f"Experiment runtime reported by script: {results['runtime_sec']:.2f} s")
print(f"Notebook wall time for run_experiment(): {analysis_wall_time_sec:.2f} s")
print(f"Seeds: {results['seeds']}")
print(f"Total pooled measurement records: {len(records_df)}")
print(f"Columns available in measurement_records: {len(records_df.columns)}")


In [ ]:
config_rows = [
    ('script_path', config['script_path']),
    ('dim', config['dim']),
    ('num_layers', config['num_layers']),
    ('n_params', config['n_params']),
    ('measurement_steps', config['measurement_steps']),
    ('ns_k_values', config['ns_k_values']),
    ('train_lr', config['train_lr']),
    ('momentum', config['momentum']),
    ('hessian_eps', config['hessian_eps']),
    ('num_seeds', config['num_seeds']),
]

runtime_rows = [
    ('script_runtime_sec', results['runtime_sec']),
    ('notebook_wall_time_sec', analysis_wall_time_sec),
    ('pooled_measurements', summary['n_measurements']),
]

direction_note_rows = [
    ('k=0 label', direction_labels_by_k.get(0, 'not present in this run')),
    (
        f"largest-k label (k={max(config['ns_k_values'])})",
        direction_labels_by_k[max(config['ns_k_values'])],
    ),
    ('k=0 note', direction_metadata['direction_notes']['k_zero_definition']),
    ('largest-k note', direction_metadata['direction_notes']['high_k_proxy_definition']),
    ('SGD baseline', baseline_labels['sgd']),
    ('Adam baseline', baseline_labels['adam']),
]

print('Configuration summary')
display(pd.DataFrame(config_rows, columns=['field', 'value']))

print('Runtime / sample-count summary')
display(pd.DataFrame(runtime_rows, columns=['field', 'value']))

print('Direction / baseline semantics used by the script')
display(pd.DataFrame(direction_note_rows, columns=['field', 'value']))

print('Seed schedule')
display(pd.DataFrame({'seed': results['seeds']}))

print('First few flattened measurement records')
display(records_df.head().round(6))


## Overall pooled alignment versus Newton–Schulz iteration count

Interpretation notes before looking at the outputs:

- Higher cosine means closer directional alignment to the pseudoinverse Newton step.
- `k=0` here is **not** the raw flat SGD direction; it is the **layerwise Frobenius-normalized gradient** with zero Newton–Schulz iterations.
- The SGD and Adam baselines are shown separately.
- `k=20` is treated only as a **high-k Newton–Schulz proxy** unless orthogonality diagnostics indicate near-convergence.


In [ ]:
overall_table = pd.DataFrame([
    {
        'k': k,
        'mean_cosine': summary['overall_by_k'][k]['mean'],
        'std_cosine': summary['overall_by_k'][k]['std'],
        'sem_cosine': summary['overall_by_k'][k]['sem'],
        'ci95_halfwidth': summary['overall_by_k'][k]['ci95_halfwidth'],
        'mean_orthogonality_residual': summary['overall_orthogonality_by_k'][k]['mean'],
    }
    for k in config['ns_k_values']
])

baseline_table = pd.DataFrame([
    {
        'baseline': name,
        'mean_cosine': summary['overall_baselines'][name]['mean'],
        'std_cosine': summary['overall_baselines'][name]['std'],
        'sem_cosine': summary['overall_baselines'][name]['sem'],
        'ci95_halfwidth': summary['overall_baselines'][name]['ci95_halfwidth'],
    }
    for name in ['sgd', 'adam']
])

print('Overall pooled NS-by-k summary')
display(overall_table.round(6))

print('Overall pooled baseline summary')
display(baseline_table.round(6))


In [ ]:
ks = overall_table['k'].to_numpy()
means = overall_table['mean_cosine'].to_numpy()
ci95 = overall_table['ci95_halfwidth'].to_numpy()

fig, ax = plt.subplots(figsize=(8.5, 5.0))
ax.errorbar(ks, means, yerr=ci95, fmt='o-', linewidth=2, capsize=4, label='NS-transformed direction')

for baseline_name, color in [('sgd', 'tab:green'), ('adam', 'tab:red')]:
    baseline_mean = summary['overall_baselines'][baseline_name]['mean']
    baseline_ci = summary['overall_baselines'][baseline_name]['ci95_halfwidth']
    ax.axhline(baseline_mean, color=color, linestyle='--', linewidth=2, label=baseline_labels[baseline_name])
    ax.fill_between([ks.min(), ks.max()], baseline_mean - baseline_ci, baseline_mean + baseline_ci, color=color, alpha=0.12)

ax.set_xlabel('Newton–Schulz iteration count k')
ax.set_ylabel('Mean cosine with pseudoinverse Newton direction')
ax.set_title('Overall pooled directional alignment versus k')
ax.legend(loc='best')
show_figure(fig)


In [ ]:
story_support = 'supports' if verdict['inverted_u_story_supported'] else 'does not support'
peak_range_text = 'inside' if verdict['pooled_peak_in_predicted_range'] else 'outside'

interpretation_md = f'''
### Interpretation of the pooled-k curve

- The pooled peak occurs at **k={verdict['pooled_peak_k']}**, which lies **{peak_range_text}** the originally expected `k=3–5` range.
- The current pooled run **{story_support}** the originally expected inverted-U story.
- Quantitatively, `k=5 - k=20 = {verdict['pooled_k5_minus_k20']:+.6f}` on the pooled cosine metric.
- Even when the inverted-U story is not supported, `k=5` can still be compared fairly with the explicit baselines:
  - `k=5 - SGD = {verdict['pooled_k5_minus_sgd']:+.6f}`
  - `k=5 - Adam = {verdict['pooled_k5_minus_adam']:+.6f}`
- This figure should be read as **directional alignment evidence** only; it does not identify the causal mechanism behind any alignment differences.
'''

display(Markdown(interpretation_md))


## Per-step behavior for selected directions

Pooling over all checkpoints is compact but can hide training-stage variation. The next figure shows the per-step means for three NS settings together with SGD and Adam snapshot baselines.


In [ ]:
steps = sorted(summary['per_step'])
selected_series = {}
for k in selected_k_values:
    selected_series[direction_labels_by_k[k]] = (
        [summary['per_step'][step]['by_k'][k]['mean'] for step in steps],
        [summary['per_step'][step]['by_k'][k]['ci95_halfwidth'] for step in steps],
    )
for baseline_name in ['sgd', 'adam']:
    selected_series[baseline_labels[baseline_name]] = (
        [summary['per_step'][step]['baselines'][baseline_name]['mean'] for step in steps],
        [summary['per_step'][step]['baselines'][baseline_name]['ci95_halfwidth'] for step in steps],
    )

fig, ax = plt.subplots(figsize=(9.0, 5.5))
for label, (series_mean, series_ci) in selected_series.items():
    ax.errorbar(steps, series_mean, yerr=series_ci, marker='o', linewidth=2, capsize=3, label=label)

ax.set_xscale('log')
ax.set_xlabel('Training step (log scale)')
ax.set_ylabel('Mean cosine with pseudoinverse Newton direction')
ax.set_title('Per-step cosine summaries for selected NS counts and baselines')
ax.legend(loc='best')
show_figure(fig)

per_step_selected_df = pd.DataFrame([
    {
        'step': step,
        **{f"k{k}_mean": summary['per_step'][step]['by_k'][k]['mean'] for k in selected_k_values},
        'sgd_mean': summary['per_step'][step]['baselines']['sgd']['mean'],
        'adam_mean': summary['per_step'][step]['baselines']['adam']['mean'],
    }
    for step in steps
])

print('Per-step mean cosine summary (selected directions)')
display(per_step_selected_df.round(6))

per_step_interpretation = f"""
### Interpretation of the per-step curves

- The ordering of directions is **not uniform across training steps**; some checkpoints favor lower `k`, while others favor the larger high-k proxy.
- This is why the pooled conclusion should be stated carefully: it summarizes the whole run, but it does not imply that `k={verdict['pooled_peak_k']}` dominates at every checkpoint.
- The explicit SGD and Adam snapshot baselines remain useful anchors for judging whether an NS-transformed direction is merely different from, or actually more aligned than, conventional first-order directions on the same trajectory.
"""

display(Markdown(per_step_interpretation))


## Hessian and training diagnostics

The script now exposes a few small diagnostics that are useful for calibration:

- loss,
- gradient norm,
- effective Hessian rank at the 1% spectral threshold,
- counts of positive / negative / near-zero eigenvalues,
- finite-difference Hessian symmetry residual before explicit symmetrization.

These are still toy-model diagnostics, but they make the experiment materially easier to interpret than a cosine-only printout.


In [ ]:
diagnostics_df = pd.DataFrame([
    {
        'step': step,
        'loss_mean': summary['per_step'][step]['diagnostics']['loss']['mean'],
        'grad_norm_mean': summary['per_step'][step]['diagnostics']['grad_norm']['mean'],
        'newton_norm_mean': summary['per_step'][step]['diagnostics']['newton_norm']['mean'],
        'hessian_effective_rank_1pct_mean': summary['per_step'][step]['diagnostics']['hessian_effective_rank_1pct']['mean'],
        'hessian_n_pos_mean': summary['per_step'][step]['diagnostics']['hessian_n_pos']['mean'],
        'hessian_n_neg_mean': summary['per_step'][step]['diagnostics']['hessian_n_neg']['mean'],
        'hessian_n_zero_tol_mean': summary['per_step'][step]['diagnostics']['hessian_n_zero_tol']['mean'],
        'abs_spectral_cond_proxy_mean': summary['per_step'][step]['diagnostics']['abs_spectral_cond_proxy']['mean'],
        'hessian_symmetry_residual_before_sym_mean': summary['per_step'][step]['diagnostics']['hessian_symmetry_residual_before_sym']['mean'],
    }
    for step in steps
])

print('Per-step Hessian and training diagnostics')
display(diagnostics_df.round(6))

fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(diagnostics_df['step'], diagnostics_df['hessian_effective_rank_1pct_mean'], marker='o', linewidth=2, label='effective rank (1% threshold)')
ax.plot(diagnostics_df['step'], diagnostics_df['hessian_n_pos_mean'], marker='s', linewidth=2, label='positive eigenvalue count')
ax.plot(diagnostics_df['step'], diagnostics_df['hessian_n_neg_mean'], marker='^', linewidth=2, label='negative eigenvalue count')
ax.plot(diagnostics_df['step'], diagnostics_df['hessian_n_zero_tol_mean'], marker='d', linewidth=2, label='near-zero count')
ax.set_xscale('log')
ax.set_xlabel('Training step (log scale)')
ax.set_ylabel('Mean count across seeds')
ax.set_title('Per-step Hessian spectrum diagnostics')
ax.legend(loc='best')
show_figure(fig)

diagnostics_interpretation = """
### Interpretation of the diagnostics

- The effective Hessian rank generally falls over training, which is consistent with the toy system entering progressively flatter local geometry.
- Positive and negative eigenvalue counts coexist for much of the trajectory, so the pseudoinverse Newton direction is being computed in an indefinite-curvature setting rather than a uniformly positive-definite one.
- These diagnostics calibrate the alignment metric, but they do **not** by themselves identify a symmetry/gauge mechanism.
"""

display(Markdown(diagnostics_interpretation))


## Orthogonality residual by Newton–Schulz iteration count

A small additional diagnostic is now available: the mean row-orthogonality residual
`||XXᵀ - I||_F / sqrt(dim)` of the layerwise NS-transformed gradient matrices. This helps justify the language of *higher-k approximation to an orthogonal factor* without over-claiming that `k=20` is exact.


In [ ]:
orth_df = pd.DataFrame([
    {
        'k': k,
        'label': direction_labels_by_k[k],
        'mean_orthogonality_residual': summary['overall_orthogonality_by_k'][k]['mean'],
        'ci95_halfwidth': summary['overall_orthogonality_by_k'][k]['ci95_halfwidth'],
    }
    for k in config['ns_k_values']
])

print('Overall orthogonality residual summary by k')
display(orth_df.round(6))

fig, ax = plt.subplots(figsize=(8.0, 4.5))
ax.errorbar(
    orth_df['k'],
    orth_df['mean_orthogonality_residual'],
    yerr=orth_df['ci95_halfwidth'],
    marker='o',
    linewidth=2,
    capsize=4,
)
ax.set_xlabel('Newton–Schulz iteration count k')
ax.set_ylabel('Mean orthogonality residual')
ax.set_title('Orthogonality residual decreases as k increases')
show_figure(fig)

orth_interpretation = f"""
### Interpretation of the orthogonality residuals

- Residuals decrease materially as `k` increases, so the larger-k transforms are in fact moving closer to an orthogonal-factor proxy.
- However, the mean residual is still **{verdict['orthogonality_residual_k20_mean']:.6f}** at `k=20`, so this notebook still avoids calling `k=20` exact orthogonalization.
- This diagnostic therefore supports the phrasing "higher-k proxy" rather than "exact ortho."
"""

display(Markdown(orth_interpretation))


In [ ]:
conclusion_md = f"""
## Calibrated conclusion

This notebook now matches the script by **importing and calling** `run_experiment()` from `run_experiment.py`, so the experimental core lives in one place.

For the default toy study

- seeds: `{results['seeds']}`
- measurement steps: `{config['measurement_steps']}`
- pooled checkpoints: `{summary['n_measurements']}`

the current run yields the following headline result:

- pooled peak cosine at **k={verdict['pooled_peak_k']}** with mean **{verdict['pooled_peak_mean_cosine']:+.6f}**,
- pooled `k=5` mean cosine **{verdict['pooled_k5_mean_cosine']:+.6f}**,
- pooled `k=20` mean cosine **{verdict['pooled_k20_mean_cosine']:+.6f}**,
- pooled difference `k=5 - k=20 = {verdict['pooled_k5_minus_k20']:+.6f}`,
- mean orthogonality residuals `k=5 -> {verdict['orthogonality_residual_k5_mean']:.6f}` and `k=20 -> {verdict['orthogonality_residual_k20_mean']:.6f}`.

Therefore, the current default run **{'supports' if verdict['inverted_u_story_supported'] else 'does not support'}** the originally expected inverted-U story.

A careful reading is:

1. this toy metric can still show whether some NS counts align better with the pseudoinverse Newton direction than others,
2. the current result should be reported as an **alignment observation**,
3. stronger claims about natural-gradient equivalence, gauge removal, or exact orthogonalization are **not justified by the present metrics alone**.

That is the appropriate first-pass conclusion for this pair of files.
"""

display(Markdown(conclusion_md))
